In [0]:
q = lambda sql: spark.sql(sql).first()[0]
T = "workspace.equipment"
checks = {
 "100 engines in train":        q(f"SELECT COUNT(DISTINCT unit) FROM {T}.silver_train") == 100,
 "no duplicate (unit, cycle)":  q(f"SELECT COUNT(*) - COUNT(DISTINCT unit, cycle) FROM {T}.silver_train") == 0,
 "train RUL min = 0":           q(f"SELECT MIN(rul) FROM {T}.silver_train") == 0,
 "no null sensors in silver":   q(f"SELECT COUNT(*) FROM {T}.silver_train WHERE s2 IS NULL OR s7 IS NULL") == 0,
 "test RUL never negative":     q(f"SELECT MIN(rul) FROM {T}.silver_test") >= 0,
 "health rows = 100":           q(f"SELECT COUNT(*) FROM {T}.gold_engine_health") == 100,
 "model RMSE sane (< 40)":      q(f"SELECT SQRT(AVG(POW(abs_error,2))) FROM {T}.gold_engine_health") < 40,
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)
assert all(checks.values())